<a href="https://colab.research.google.com/github/islam37/S-SLTE-Thesis/blob/main/notebooks/06_matched_rate_protocol.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

import json
import pandas as pd
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")

BASE = Path("/content/drive/MyDrive/S_SLTE_Thesis")

# Load existing frozen dataset partitions
anchor = pd.read_csv(BASE / "data/splits/anchor.csv")
development = pd.read_csv(
    BASE / "data/splits/development.csv"
)
noisy_training = pd.read_csv(
    BASE / "data/splits/noisy_training.csv"
)
pilot = pd.read_csv(
    BASE / "data/pilot/pilot_500.csv"
)

# Verify known dataset sizes
assert len(anchor) == 2000
assert len(development) == 1800
assert len(noisy_training) == 16200
assert len(pilot) == 500

# Verify unique IDs
for name, df in [
    ("anchor", anchor),
    ("development", development),
    ("noisy_training", noisy_training),
    ("pilot", pilot)
]:
    assert df["record_id"].is_unique, (
        f"Duplicate record IDs in {name}"
    )

# Verify partition separation
anchor_ids = set(anchor["record_id"])
dev_ids = set(development["record_id"])
noisy_ids = set(noisy_training["record_id"])
pilot_ids = set(pilot["record_id"])

assert anchor_ids.isdisjoint(dev_ids)
assert anchor_ids.isdisjoint(noisy_ids)
assert dev_ids.isdisjoint(noisy_ids)

# Pilot must be a subset of noisy training
assert pilot_ids.issubset(noisy_ids)

# Check required labels
assert "label" in anchor.columns
assert "label" in development.columns

# Do not permit reference labels in noisy inputs
assert "label" not in noisy_training.columns
assert "label" not in pilot.columns

assert anchor["label"].isin(range(4)).all()
assert development["label"].isin(range(4)).all()

# Verify pilot text matches original noisy-training text
pilot_check = pilot.merge(
    noisy_training,
    on="record_id",
    how="left",
    validate="one_to_one",
    suffixes=("_pilot", "_training")
)

assert len(pilot_check) == 500
assert (
    pilot_check["text_pilot"] ==
    pilot_check["text_training"]
).all()

print("DATASET INTEGRITY CHECK PASSED")
print("--------------------------------")
print("Clean anchor:", len(anchor))
print("Development:", len(development))
print("Noisy training:", len(noisy_training))
print("Exploratory pilot:", len(pilot))
print("Partition overlap: NONE")
print("Pilot subset verification: PASSED")
print("Reference labels in noisy input: NONE")

# Define experiment protocol DRAFT
# No real baseline is executed here.

protocol = {
    "protocol_id": "matched_rate_noise_draft_v1",
    "status": "DRAFT_REQUIRES_REVIEW",

    "dataset": "AG News",
    "num_classes": 4,

    "partitions": {
        "clean_anchor": 2000,
        "development": 1800,
        "noisy_training_pool": 16200,
        "exploratory_pilot": 500,
        "official_test": 7600
    },

    "noise_conditions": {
        "primary_synthetic_rates": [
            0.10, 0.20, 0.30
        ],
        "pilot_exploratory_rate": 0.218,
        "note": (
            "21.8% is observed pilot disagreement, "
            "not a verified full-dataset noise rate"
        )
    },

    "symmetric_noise": {
        "type": "fixed_count",
        "replacement": "uniform_incorrect_class",
        "seeds": [42, 123, 2026],
        "clean_label_access": (
            "isolated baseline data-construction "
            "stage only"
        )
    },

    "evaluation_design": {
        "same_records_for_comparison": True,
        "keep_pilot_exploratory": True,
        "proposed_holdout_size": 2000,
        "holdout_source": (
            "noisy training pool excluding pilot"
        ),
        "holdout_selection_seed": 20261025,
        "holdout_status": (
            "PROPOSED_NOT_CREATED"
        ),
        "threshold_tuning_source": "development",
        "official_test_policy": (
            "final downstream evaluation only"
        ),
        "restriction": (
            "Never pass reference labels to "
            "the S-SLTE correction pipeline"
        )
    },

    "planned_metrics": [
        "noise_detection_precision",
        "noise_detection_recall",
        "noise_detection_f1",
        "correction_precision",
        "harmful_correction_share",
        "coverage",
        "accepted_error_rate",
        "downstream_accuracy",
        "downstream_macro_f1"
    ],

    "next_stage": (
        "review holdout design and implement "
        "data-isolated experimental partitions"
    )
}

CONFIG_DIR = BASE / "configs"
CONFIG_DIR.mkdir(parents=True, exist_ok=True)

output_path = (
    CONFIG_DIR /
    "matched_rate_noise_protocol_draft_v1.json"
)

# Preserve any existing version of the draft
if output_path.exists():
    with open(output_path, encoding="utf-8") as f:
        existing = json.load(f)

    assert existing == protocol, (
        "Existing protocol differs. "
        "Manual review required."
    )

    print("EXISTING MATCHED-RATE PROTOCOL VERIFIED")

else:
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(
            protocol,
            f,
            indent=2,
            ensure_ascii=False
        )

    print("MATCHED-RATE PROTOCOL DRAFT SAVED")

print("Status:", protocol["status"])
print("Proposed holdout: 2000 (not created)")
print("Real baseline experiments executed: NO")


Mounted at /content/drive
DATASET INTEGRITY CHECK PASSED
--------------------------------
Clean anchor: 2000
Development: 1800
Noisy training: 16200
Exploratory pilot: 500
Partition overlap: NONE
Pilot subset verification: PASSED
Reference labels in noisy input: NONE
MATCHED-RATE PROTOCOL DRAFT SAVED
Status: DRAFT_REQUIRES_REVIEW
Proposed holdout: 2000 (not created)
Real baseline experiments executed: NO


In [2]:

import json
import hashlib
import numpy as np
import pandas as pd
from pathlib import Path

BASE = Path("/content/drive/MyDrive/S_SLTE_Thesis")

with open(
    BASE / "configs/matched_rate_noise_protocol_draft_v1.json",
    encoding="utf-8"
) as f:
    protocol = json.load(f)

assert protocol["status"] == "DRAFT_REQUIRES_REVIEW"

# Reload only permitted ID and text data
noisy = pd.read_csv(
    BASE / "data/splits/noisy_training.csv"
)

pilot = pd.read_csv(
    BASE / "data/pilot/pilot_500.csv"
)

anchor = pd.read_csv(
    BASE / "data/splits/anchor.csv"
)

development = pd.read_csv(
    BASE / "data/splits/development.csv"
)

# Never load private reference labels here.
assert "label" not in noisy.columns
assert "label" not in pilot.columns

SEED = 20261025
HOLDOUT_SIZE = 2000

# Stable ordering independent of CSV row order
noisy_ids = set(noisy["record_id"].astype(int))
pilot_ids = set(pilot["record_id"].astype(int))

assert len(noisy_ids) == 16200
assert len(pilot_ids) == 500
assert pilot_ids.issubset(noisy_ids)

available_ids = np.array(
    sorted(noisy_ids - pilot_ids),
    dtype=np.int64
)

assert len(available_ids) == 15700

# Deterministic proposed holdout selection
rng = np.random.default_rng(SEED)

holdout_ids = set(
    rng.choice(
        available_ids,
        size=HOLDOUT_SIZE,
        replace=False
    ).tolist()
)

# Remaining training IDs
training_ids = (
    noisy_ids - pilot_ids - holdout_ids
)

# Verify counts
assert len(holdout_ids) == 2000
assert len(training_ids) == 13700

# Verify no overlap
assert holdout_ids.isdisjoint(training_ids)
assert holdout_ids.isdisjoint(pilot_ids)
assert training_ids.isdisjoint(pilot_ids)

anchor_ids = set(anchor["record_id"].astype(int))
dev_ids = set(development["record_id"].astype(int))

for selected_ids in [holdout_ids, training_ids]:
    assert selected_ids.isdisjoint(anchor_ids)
    assert selected_ids.isdisjoint(dev_ids)

# Verify all records are accounted for
assert (
    holdout_ids | training_ids | pilot_ids
) == noisy_ids

# Verify repeatability with the same seed
rng_again = np.random.default_rng(SEED)

holdout_repeat = set(
    rng_again.choice(
        available_ids,
        size=HOLDOUT_SIZE,
        replace=False
    ).tolist()
)

assert holdout_ids == holdout_repeat

# Produce a deterministic selection fingerprint
serialized_ids = "\n".join(
    str(x) for x in sorted(holdout_ids)
).encode("utf-8")

holdout_fingerprint = hashlib.sha256(
    serialized_ids
).hexdigest()

print("HOLDOUT DESIGN VALIDATION PASSED")
print("--------------------------------")
print("Original noisy pool:", len(noisy_ids))
print("Exploratory pilot:", len(pilot_ids))
print("Proposed holdout:", len(holdout_ids))
print("Remaining training:", len(training_ids))
print("Partition overlap: NONE")
print("Reproducibility: PASSED")
print("Reference labels accessed: NO")
print("Original files modified: NO")
print("Holdout saved to Drive: NO")
print("Selection SHA256:", holdout_fingerprint)

print("\nStatus: PROPOSED — NOT YET FROZEN")


HOLDOUT DESIGN VALIDATION PASSED
--------------------------------
Original noisy pool: 16200
Exploratory pilot: 500
Proposed holdout: 2000
Remaining training: 13700
Partition overlap: NONE
Reproducibility: PASSED
Reference labels accessed: NO
Original files modified: NO
Holdout saved to Drive: NO
Selection SHA256: 37bd15b5fb600e7cfc392c936fcb187b0a39313c3502098c0e6a403153057051

Status: PROPOSED — NOT YET FROZEN


In [3]:

import hashlib
import pandas as pd

# Uses the dataframes and ID sets from Step 2.
# Does not read reference labels or modify any files.

def text_fingerprints(df):
    assert {"record_id", "text"}.issubset(df.columns)
    assert df["text"].notna().all()

    normalized = (
        df["text"]
        .astype(str)
        .str.normalize("NFKC")
        .str.casefold()
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )

    assert normalized.ne("").all()

    return normalized.map(
        lambda text: hashlib.sha256(
            text.encode("utf-8")
        ).hexdigest()
    )

# Build separate, non-overlapping proposed groups
training_df = noisy[
    noisy["record_id"].isin(training_ids)
].copy()

holdout_df = noisy[
    noisy["record_id"].isin(holdout_ids)
].copy()

groups = {
    "anchor": anchor,
    "development": development,
    "training": training_df,
    "pilot": pilot,
    "holdout": holdout_df
}

assert len(training_df) == 13700
assert len(holdout_df) == 2000

fingerprints = {
    name: text_fingerprints(df)
    for name, df in groups.items()
}

# Check duplicate content inside each partition
print("INTERNAL DUPLICATE AUDIT")
print("------------------------")

for name, hashes in fingerprints.items():
    duplicates = int(hashes.duplicated().sum())
    print(f"{name}: {duplicates} duplicate rows")

# Check identical normalized text across groups
print("\nCROSS-PARTITION TEXT OVERLAP")
print("----------------------------")

group_names = list(groups)

overlap_results = []

for i, name_a in enumerate(group_names):
    for name_b in group_names[i + 1:]:

        hashes_a = set(fingerprints[name_a])
        hashes_b = set(fingerprints[name_b])

        shared = hashes_a & hashes_b

        overlap_results.append({
            "group_a": name_a,
            "group_b": name_b,
            "shared_unique_texts": len(shared)
        })

overlap_df = pd.DataFrame(overlap_results)

display(overlap_df)

# Holdout-specific contamination indicators
holdout_hashes = set(fingerprints["holdout"])

for name in ["anchor", "development", "training", "pilot"]:
    shared = (
        holdout_hashes &
        set(fingerprints[name])
    )

    print(
        f"Holdout vs {name}: "
        f"{len(shared)} shared unique texts"
    )

# Summary — observation, not automatic dataset change
total_cross_overlaps = int(
    overlap_df["shared_unique_texts"].sum()
)

print("\nTEXT LEAKAGE AUDIT COMPLETED")

if total_cross_overlaps == 0:
    print("Normalized exact cross-partition overlaps: NONE")
else:
    print(
        "Normalized exact cross-partition overlaps found:",
        total_cross_overlaps
    )
    print("REVIEW REQUIRED BEFORE HOLDOUT FREEZE")

print("Reference labels accessed: NO")
print("Dataset files modified: NO")
print("Holdout status: PROPOSED_NOT_FROZEN")


INTERNAL DUPLICATE AUDIT
------------------------
anchor: 0 duplicate rows
development: 0 duplicate rows
training: 2 duplicate rows
pilot: 0 duplicate rows
holdout: 0 duplicate rows

CROSS-PARTITION TEXT OVERLAP
----------------------------


,group_a,group_b,shared_unique_texts
0,anchor,development,0
1,anchor,training,1
2,anchor,pilot,0
3,anchor,holdout,0
4,development,training,1
5,development,pilot,0
6,development,holdout,0
7,training,pilot,0
8,training,holdout,0
9,pilot,holdout,0


Holdout vs anchor: 0 shared unique texts
Holdout vs development: 0 shared unique texts
Holdout vs training: 0 shared unique texts
Holdout vs pilot: 0 shared unique texts

TEXT LEAKAGE AUDIT COMPLETED
Normalized exact cross-partition overlaps found: 2
REVIEW REQUIRED BEFORE HOLDOUT FREEZE
Reference labels accessed: NO
Dataset files modified: NO
Holdout status: PROPOSED_NOT_FROZEN


In [4]:

import pandas as pd
import hashlib

# Reuse groups and text_fingerprints()
# from the previous audit cell.

audit_rows = []

for group_name, group_df in groups.items():

    hashes = text_fingerprints(group_df)

    for record_id, text_hash in zip(
        group_df["record_id"].tolist(),
        hashes.tolist()
    ):
        audit_rows.append({
            "partition": group_name,
            "record_id": int(record_id),
            "text_hash": text_hash
        })

audit_df = pd.DataFrame(audit_rows)

# A repeated normalized text hash identifies
# a possible duplicate group.
duplicate_groups = (
    audit_df.groupby("text_hash")
    .agg(
        record_count=("record_id", "size"),
        partitions=(
            "partition",
            lambda x: sorted(set(x))
        )
    )
    .reset_index()
)

duplicate_groups = duplicate_groups[
    duplicate_groups["record_count"] > 1
].copy()

print("DUPLICATE GROUP IDENTIFICATION")
print("------------------------------")

print(
    "Unique duplicated text groups:",
    len(duplicate_groups)
)

for index, row in enumerate(
    duplicate_groups.itertuples(index=False),
    start=1
):
    matching = audit_df[
        audit_df["text_hash"] == row.text_hash
    ].sort_values(["partition", "record_id"])

    print(f"\nDuplicate Group {index}")
    print("Hash prefix:", row.text_hash[:16])
    print("Record count:", row.record_count)

    # No article text or reference labels displayed
    display(
        matching[["partition", "record_id"]]
        .reset_index(drop=True)
    )

# Preview a possible training-only cleanup.
# DO NOT save or modify anything.

protected_partitions = [
    "anchor",
    "development",
    "pilot",
    "holdout"
]

protected_hashes = set(
    audit_df.loc[
        audit_df["partition"].isin(
            protected_partitions
        ),
        "text_hash"
    ]
)

training_audit = audit_df[
    audit_df["partition"] == "training"
].copy()

# Identify training records colliding with
# protected partitions.
colliding = training_audit[
    training_audit["text_hash"].isin(
        protected_hashes
    )
].copy()

# Preview training deduplication.
remaining = training_audit[
    ~training_audit["text_hash"].isin(
        protected_hashes
    )
].copy()

remaining_unique = remaining.drop_duplicates(
    subset="text_hash",
    keep="first"
)

removed_total = (
    len(training_audit) -
    len(remaining_unique)
)

assert (
    remaining_unique["text_hash"].is_unique
)

assert not (
    set(remaining_unique["text_hash"])
    & protected_hashes
)

print("\nPROPOSED CLEANUP PREVIEW")
print("------------------------")
print(
    "Training rows before:",
    len(training_audit)
)
print(
    "Rows colliding with protected partitions:",
    len(colliding)
)
print(
    "Training rows after preview:",
    len(remaining_unique)
)
print(
    "Total rows excluded in preview:",
    removed_total
)

print("\nDUPLICATE DIAGNOSTIC COMPLETED")
print("Original dataset modified: NO")
print("Reference labels accessed: NO")
print("Holdout status: NOT FROZEN")


DUPLICATE GROUP IDENTIFICATION
------------------------------
Unique duplicated text groups: 4

Duplicate Group 1
Hash prefix: 4cc72065836afd9f
Record count: 2


,partition,record_id
0,anchor,5287
1,training,9266



Duplicate Group 2
Hash prefix: 6ded4191a3665b07
Record count: 2


,partition,record_id
0,training,66753
1,training,68708



Duplicate Group 3
Hash prefix: 6e58969cc67c036b
Record count: 2


,partition,record_id
0,development,42966
1,training,43231



Duplicate Group 4
Hash prefix: dc33c6e48e827d5d
Record count: 2


,partition,record_id
0,training,79471
1,training,79605



PROPOSED CLEANUP PREVIEW
------------------------
Training rows before: 13700
Rows colliding with protected partitions: 2
Training rows after preview: 13696
Total rows excluded in preview: 4

DUPLICATE DIAGNOSTIC COMPLETED
Original dataset modified: NO
Reference labels accessed: NO
Holdout status: NOT FROZEN


In [5]:

import pandas as pd
import hashlib

# Reuse training_df, groups and
# text_fingerprints() from previous cells.

# Protected datasets must remain unchanged.
protected_names = [
    "anchor",
    "development",
    "pilot",
    "holdout"
]

protected_hashes = set()

for name in protected_names:
    protected_hashes.update(
        text_fingerprints(groups[name]).tolist()
    )

# Sort by record_id for deterministic selection
candidate = training_df.sort_values(
    "record_id"
).copy()

candidate["text_hash"] = (
    text_fingerprints(candidate).to_numpy()
)

assert len(candidate) == 13700
assert candidate["record_id"].is_unique

# Remove training records duplicating
# any protected partition.
protected_collision = candidate[
    candidate["text_hash"].isin(protected_hashes)
].copy()

without_collisions = candidate[
    ~candidate["text_hash"].isin(protected_hashes)
].copy()

# Within training, retain the smallest record_id
# for each normalized text hash.
internal_duplicates = without_collisions[
    without_collisions.duplicated(
        subset="text_hash",
        keep="first"
    )
].copy()

clean_training = without_collisions.drop_duplicates(
    subset="text_hash",
    keep="first"
).copy()

# Build an exclusion audit (IDs only)
removed = pd.concat(
    [
        protected_collision.assign(
            exclusion_reason="protected_overlap"
        ),
        internal_duplicates.assign(
            exclusion_reason="internal_duplicate"
        )
    ],
    ignore_index=True
)

removed = removed[
    ["record_id", "exclusion_reason", "text_hash"]
].sort_values("record_id")

# Expected exclusions based on today's diagnosis
expected_removed_ids = {
    9266,
    68708,
    43231,
    79605
}

actual_removed_ids = set(
    removed["record_id"].astype(int)
)

assert actual_removed_ids == expected_removed_ids

# Essential invariants
assert len(clean_training) == 13696
assert len(removed) == 4

assert clean_training["record_id"].is_unique
assert clean_training["text_hash"].is_unique

assert not (
    set(clean_training["text_hash"])
    & protected_hashes
)

original_ids = set(
    training_df["record_id"].astype(int)
)

clean_ids = set(
    clean_training["record_id"].astype(int)
)

assert (
    clean_ids | actual_removed_ids
) == original_ids

assert clean_ids.isdisjoint(actual_removed_ids)

# Ensure protected partitions weren't modified
assert len(groups["anchor"]) == 2000
assert len(groups["development"]) == 1800
assert len(groups["pilot"]) == 500
assert len(groups["holdout"]) == 2000

# Fingerprint cleaned record IDs
id_bytes = "\n".join(
    str(x) for x in sorted(clean_ids)
).encode("utf-8")

clean_fingerprint = hashlib.sha256(
    id_bytes
).hexdigest()

print("DEDUPLICATED TRAINING VIEW VALIDATED")
print("------------------------------------")
print("Original training:", len(training_df))
print("Protected overlaps:", len(protected_collision))
print("Internal duplicates:", len(internal_duplicates))
print("Excluded records:", len(removed))
print("Cleaned training:", len(clean_training))

print("\nEXCLUSION AUDIT")
display(
    removed[
        ["record_id", "exclusion_reason"]
    ].reset_index(drop=True)
)

print("\nClean training ID SHA256:")
print(clean_fingerprint)

print("\nOriginal dataset modified: NO")
print("Google Drive files written: NO")
print("Reference labels accessed: NO")
print("Holdout status: PROPOSED_NOT_FROZEN")
print("Training view status: CANDIDATE_NOT_FROZEN")


DEDUPLICATED TRAINING VIEW VALIDATED
------------------------------------
Original training: 13700
Protected overlaps: 2
Internal duplicates: 2
Excluded records: 4
Cleaned training: 13696

EXCLUSION AUDIT


,record_id,exclusion_reason
0,9266,protected_overlap
1,43231,protected_overlap
2,68708,internal_duplicate
3,79605,internal_duplicate



Clean training ID SHA256:
52176ba0a8d8d43415ff34c083f75de26f9db05b73458893a23413998aa28b88

Original dataset modified: NO
Google Drive files written: NO
Reference labels accessed: NO
Holdout status: PROPOSED_NOT_FROZEN
Training view status: CANDIDATE_NOT_FROZEN


In [6]:

import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Reuse previous verified dataframes.
# Never load private reference labels here.

holdout_check = holdout_df[
    ["record_id", "text"]
].copy().reset_index(drop=True)

comparison_parts = []

for name, source in [
    ("anchor", anchor),
    ("development", development),
    ("training", clean_training),
    ("pilot", pilot)
]:
    part = source[["record_id", "text"]].copy()
    part["partition"] = name
    comparison_parts.append(part)

comparison = pd.concat(
    comparison_parts,
    ignore_index=True
)

assert len(holdout_check) == 2000
assert len(comparison) == 17996
assert holdout_check["text"].notna().all()
assert comparison["text"].notna().all()

assert set(
    holdout_check["record_id"]
).isdisjoint(
    set(comparison["record_id"])
)

# Fit TF-IDF on texts only, not labels.
vectorizer = TfidfVectorizer(
    lowercase=True,
    strip_accents="unicode",
    ngram_range=(1, 2),
    min_df=2,
    max_features=80000,
    sublinear_tf=True,
    dtype=np.float32
)

all_texts = pd.concat(
    [
        holdout_check["text"],
        comparison["text"]
    ],
    ignore_index=True
).astype(str)

X = vectorizer.fit_transform(all_texts)

X_holdout = X[:len(holdout_check)]
X_comparison = X[len(holdout_check):]

# Process in batches to limit memory usage.
BATCH_SIZE = 100

best_scores = []
best_indices = []

for start in range(0, len(holdout_check), BATCH_SIZE):
    end = min(start + BATCH_SIZE, len(holdout_check))

    similarities = cosine_similarity(
        X_holdout[start:end],
        X_comparison
    )

    match_idx = similarities.argmax(axis=1)

    match_scores = similarities[
        np.arange(len(match_idx)),
        match_idx
    ]

    best_indices.extend(match_idx.tolist())
    best_scores.extend(match_scores.tolist())

best_matches = comparison.iloc[
    best_indices
].reset_index(drop=True)

audit = pd.DataFrame({
    "holdout_record_id": holdout_check["record_id"],
    "closest_record_id": best_matches["record_id"],
    "closest_partition": best_matches["partition"],
    "cosine_similarity": best_scores
})

audit = audit.sort_values(
    "cosine_similarity",
    ascending=False
).reset_index(drop=True)

# Screening thresholds, not experimental
# hyperparameters for S-SLTE.
VERY_HIGH = 0.95
HIGH = 0.85

very_high_count = int(
    (audit["cosine_similarity"] >= VERY_HIGH).sum()
)

high_count = int(
    (audit["cosine_similarity"] >= HIGH).sum()
)

print("HOLDOUT NEAR-DUPLICATE AUDIT COMPLETED")
print("--------------------------------------")
print("Holdout records:", len(holdout_check))
print("Comparison records:", len(comparison))
print("TF-IDF features:", X.shape[1])
print("Similarity >= 0.95:", very_high_count)
print("Similarity >= 0.85:", high_count)

print("\nTOP 10 MOST SIMILAR PAIRS")
display(audit.head(10))

if high_count > 0:
    print("\nPotential near-duplicates require review.")
else:
    print("\nNo pairs exceeded the screening threshold.")

print("\nReference labels accessed: NO")
print("Google Drive files modified: NO")
print("GitHub files modified: NO")
print("Holdout frozen: NO")


HOLDOUT NEAR-DUPLICATE AUDIT COMPLETED
--------------------------------------
Holdout records: 2000
Comparison records: 17996
TF-IDF features: 80000
Similarity >= 0.95: 18
Similarity >= 0.85: 57

TOP 10 MOST SIMILAR PAIRS


,holdout_record_id,closest_record_id,closest_partition,cosine_similarity
0,71462,70932,training,1.000000
1,117637,117330,development,0.992581
2,37069,37115,training,0.991327
3,98008,97990,anchor,0.989316
4,106497,106707,training,0.987014
5,69639,69631,training,0.986044
6,19335,19190,training,0.983971
7,107847,107824,anchor,0.981942
8,55761,53252,training,0.981683
9,90178,90148,development,0.979648



Potential near-duplicates require review.

Reference labels accessed: NO
Google Drive files modified: NO
GitHub files modified: NO
Holdout frozen: NO


In [7]:

import re
import unicodedata
from difflib import SequenceMatcher
from collections import Counter

import pandas as pd

# Uses audit, holdout_check and comparison
# from the previous TF-IDF audit cell.
# No reference labels are loaded.

def normalize_for_review(text):
    text = unicodedata.normalize(
        "NFKC", str(text)
    ).casefold()

    return re.sub(r"\s+", " ", text).strip()


def jaccard_similarity(text_a, text_b):
    words_a = set(text_a.split())
    words_b = set(text_b.split())

    union = words_a | words_b

    if not union:
        return 1.0

    return len(words_a & words_b) / len(union)


# Review all 57 pairs above the screening threshold
candidates = audit[
    audit["cosine_similarity"] >= 0.85
].copy()

holdout_lookup = holdout_check.set_index(
    "record_id"
)["text"]

comparison_lookup = comparison.set_index(
    "record_id"
)["text"]

assert holdout_lookup.index.is_unique
assert comparison_lookup.index.is_unique

review_rows = []

for row in candidates.itertuples(index=False):

    text_a = normalize_for_review(
        holdout_lookup.loc[row.holdout_record_id]
    )

    text_b = normalize_for_review(
        comparison_lookup.loc[row.closest_record_id]
    )

    sequence_score = SequenceMatcher(
        None,
        text_a,
        text_b,
        autojunk=False
    ).ratio()

    jaccard_score = jaccard_similarity(
        text_a,
        text_b
    )

    review_rows.append({
        "holdout_id": int(row.holdout_record_id),
        "matching_id": int(row.closest_record_id),
        "partition": row.closest_partition,
        "tfidf": round(
            row.cosine_similarity, 5
        ),
        "sequence": round(
            sequence_score, 5
        ),
        "jaccard": round(
            jaccard_score, 5
        ),
        "normalized_exact": text_a == text_b
    })

review_df = pd.DataFrame(
    review_rows
).sort_values(
    "tfidf",
    ascending=False
).reset_index(drop=True)

assert len(review_df) == 57

# Counts by partition
partition_summary = (
    review_df.groupby("partition")
    .agg(
        flagged_pairs=("holdout_id", "size"),
        max_tfidf=("tfidf", "max")
    )
    .reset_index()
)

print("CROSS-METHOD DUPLICATE AUDIT COMPLETED")
print("--------------------------------------")
print("Candidate pairs reviewed:", len(review_df))

print(
    "Normalized exact matches:",
    int(review_df["normalized_exact"].sum())
)

print("\nCANDIDATES BY PARTITION")
display(partition_summary)

print("\nTOP 10 CROSS-METHOD RESULTS")
display(review_df.head(10))

# Strong candidates according to both
# independently computed text-comparison scores.
# Screening criteria only, not automatic
# removal rules.
strong_candidates = review_df[
    (review_df["sequence"] >= 0.90) &
    (review_df["jaccard"] >= 0.85)
]

print(
    "\nStrong screening candidates:",
    len(strong_candidates)
)

print("\nReference labels accessed: NO")
print("Dataset files modified: NO")
print("Holdout frozen: NO")
print("Human review required: YES")


CROSS-METHOD DUPLICATE AUDIT COMPLETED
--------------------------------------
Candidate pairs reviewed: 57
Normalized exact matches: 0

CANDIDATES BY PARTITION


,partition,flagged_pairs,max_tfidf
0,anchor,6,0.98932
1,development,9,0.99258
2,pilot,2,0.97473
3,training,40,1.00000



TOP 10 CROSS-METHOD RESULTS


,holdout_id,matching_id,partition,tfidf,sequence,jaccard,normalized_exact
0,71462,70932,training,1.00000,0.96585,0.96429,False
1,117637,117330,development,0.99258,0.98069,0.92105,False
2,37069,37115,training,0.99133,0.97596,0.89286,False
3,98008,97990,anchor,0.98932,0.95175,0.88235,False
4,106497,106707,training,0.98701,0.95733,0.73171,False
5,69639,69631,training,0.98604,0.96564,0.80488,False
6,19335,19190,training,0.98397,0.94737,0.77273,False
7,107847,107824,anchor,0.98194,0.95522,0.94444,False
8,55761,53252,training,0.98168,0.97704,0.94872,False
9,90178,90148,development,0.97965,0.95701,0.75000,False



Strong screening candidates: 15

Reference labels accessed: NO
Dataset files modified: NO
Holdout frozen: NO
Human review required: YES


In [8]:

import pandas as pd
import re
import unicodedata

# Uses review_df, holdout_lookup and
# comparison_lookup from previous cells.
# No reference labels are accessed.

PROTECTED = ["anchor", "development", "pilot"]

# Prioritize high-similarity protected-partition pairs
priority = (
    review_df[
        review_df["partition"].isin(PROTECTED)
    ]
    .sort_values(
        "tfidf",
        ascending=False
    )
    .head(5)
    .copy()
)

assert len(priority) == 5

def readable_text(value):
    text = unicodedata.normalize(
        "NFKC",
        str(value)
    )

    return re.sub(r"\s+", " ", text).strip()

print("PRIORITY NEAR-DUPLICATE REVIEW")
print("--------------------------------")
print("Pairs selected:", len(priority))

for number, row in enumerate(
    priority.itertuples(index=False),
    start=1
):
    holdout_text = readable_text(
        holdout_lookup.loc[row.holdout_id]
    )

    matched_text = readable_text(
        comparison_lookup.loc[row.matching_id]
    )

    print("\n" + "=" * 55)
    print(f"PAIR {number}")
    print("=" * 55)

    print("Holdout ID:", row.holdout_id)
    print("Matching ID:", row.matching_id)
    print("Matching Partition:", row.partition)

    print("TF-IDF Similarity:", row.tfidf)
    print("Sequence Similarity:", row.sequence)
    print("Jaccard Similarity:", row.jaccard)

    print("\nHOLDOUT ARTICLE:")
    print(holdout_text)

    print("\nMATCHING ARTICLE:")
    print(matched_text)

    print("\nHuman classification: PENDING")

print("\nPRIORITY REVIEW DATA READY")
print("Reference labels accessed: NO")
print("Files modified: NO")
print("Holdout frozen: NO")


PRIORITY NEAR-DUPLICATE REVIEW
--------------------------------
Pairs selected: 5

PAIR 1
Holdout ID: 117637
Matching ID: 117330
Matching Partition: development
TF-IDF Similarity: 0.99258
Sequence Similarity: 0.98069
Jaccard Similarity: 0.92105

HOLDOUT ARTICLE:
Intel Acquires Chip Designers From HP (AP) AP - Intel Corp. has reached an agreement to hire hundreds of Hewlett-Packard Co. engineers who helped design the Itanium microprocessor, a massive joint project between the two technology companies since the early 1990s.

MATCHING ARTICLE:
Intel Acquires Chip Designers From HP Intel Corp. has reached an agreement to hire hundreds of Hewlett-Packard Co. engineers who helped design the Itanium microprocessor, a massive joint project between the two technology companies since the early 1990s.

Human classification: PENDING

PAIR 2
Holdout ID: 98008
Matching ID: 97990
Matching Partition: anchor
TF-IDF Similarity: 0.98932
Sequence Similarity: 0.95175
Jaccard Similarity: 0.88235

HOLDOUT AR

In [9]:

import numpy as np
import pandas as pd

# Reuse X_holdout, X_comparison,
# holdout_check and comparison
# from the previous TF-IDF audit.

TOP_K = 5
BATCH_SIZE = 100
SIMILARITY_THRESHOLD = 0.85

assert X_holdout.shape[0] == 2000
assert X_comparison.shape[0] == 17996
assert TOP_K <= X_comparison.shape[0]

expanded_rows = []

for start in range(0, X_holdout.shape[0], BATCH_SIZE):

    end = min(
        start + BATCH_SIZE,
        X_holdout.shape[0]
    )

    # TF-IDF vectors are L2 normalized.
    # Their dot product equals cosine similarity.
    similarities = (
        X_holdout[start:end] @ X_comparison.T
    ).toarray()

    top_indices = np.argpartition(
        similarities,
        -TOP_K,
        axis=1
    )[:, -TOP_K:]

    for local_idx in range(end - start):

        holdout_idx = start + local_idx

        # Order selected neighbors by similarity
        neighbors = sorted(
            top_indices[local_idx],
            key=lambda idx: similarities[
                local_idx, idx
            ],
            reverse=True
        )

        for rank, comparison_idx in enumerate(
            neighbors,
            start=1
        ):
            score = float(
                similarities[
                    local_idx, comparison_idx
                ]
            )

            if score < SIMILARITY_THRESHOLD:
                continue

            source = comparison.iloc[
                comparison_idx
            ]

            expanded_rows.append({
                "holdout_id": int(
                    holdout_check.iloc[
                        holdout_idx
                    ]["record_id"]
                ),
                "matching_id": int(
                    source["record_id"]
                ),
                "partition": str(
                    source["partition"]
                ),
                "neighbor_rank": rank,
                "tfidf_similarity": score
            })

expanded_review_df = pd.DataFrame(
    expanded_rows
)

assert not expanded_review_df.empty

assert not expanded_review_df.duplicated(
    subset=["holdout_id", "matching_id"]
).any()

# Check all previous top-1 candidates
# still appear in the expanded search.
previous_pairs = set(
    zip(
        audit.loc[
            audit["cosine_similarity"] >= 0.85,
            "holdout_record_id"
        ].astype(int),
        audit.loc[
            audit["cosine_similarity"] >= 0.85,
            "closest_record_id"
        ].astype(int)
    )
)

expanded_pairs = set(
    zip(
        expanded_review_df["holdout_id"],
        expanded_review_df["matching_id"]
    )
)

assert previous_pairs.issubset(expanded_pairs)

summary = (
    expanded_review_df.groupby("partition")
    .agg(
        flagged_pairs=("holdout_id", "size"),
        unique_holdout_ids=(
            "holdout_id", "nunique"
        ),
        max_similarity=(
            "tfidf_similarity", "max"
        )
    )
    .reset_index()
)

unique_holdouts = (
    expanded_review_df["holdout_id"].nunique()
)

multiple_matches = (
    expanded_review_df.groupby(
        "holdout_id"
    ).size()
)

print("TOP-5 NEIGHBOR AUDIT COMPLETED")
print("--------------------------------")
print("Holdout records scanned:", 2000)
print("Neighbors checked per record:", TOP_K)
print(
    "Candidate pairs >= 0.85:",
    len(expanded_review_df)
)
print(
    "Unique flagged holdout records:",
    unique_holdouts
)
print(
    "Holdout records with multiple flagged pairs:",
    int((multiple_matches > 1).sum())
)

print("\nCANDIDATES BY PARTITION")
display(summary)

print("\nTOP 15 PAIRS")
display(
    expanded_review_df.sort_values(
        "tfidf_similarity",
        ascending=False
    ).head(15).round(5)
)

print("\nPrevious 57 top-1 pairs preserved: YES")
print("Reference labels accessed: NO")
print("Files modified: NO")
print("Holdout frozen: NO")


TOP-5 NEIGHBOR AUDIT COMPLETED
--------------------------------
Holdout records scanned: 2000
Neighbors checked per record: 5
Candidate pairs >= 0.85: 63
Unique flagged holdout records: 57
Holdout records with multiple flagged pairs: 6

CANDIDATES BY PARTITION


,partition,flagged_pairs,unique_holdout_ids,max_similarity
0,anchor,7,7,0.989316
1,development,9,9,0.992581
2,pilot,2,2,0.974733
3,training,45,45,1.000000



TOP 15 PAIRS


,holdout_id,matching_id,partition,neighbor_rank,tfidf_similarity
34,71462,70932,training,1,1.00000
61,117637,117330,development,1,0.99258
21,37069,37115,training,1,0.99133
44,98008,97990,anchor,1,0.98932
50,106497,106707,training,1,0.98701
32,69639,69631,training,1,0.98604
13,19335,19190,training,1,0.98397
52,107847,107824,anchor,1,0.98194
29,55761,53252,training,1,0.98168
39,90178,90148,development,1,0.97965



Previous 57 top-1 pairs preserved: YES
Reference labels accessed: NO
Files modified: NO
Holdout frozen: NO
